In [ ]:
'''
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MINI PROJECT — DAY 10 OF 3
Hospital Data Pipeline
Phase 3: Analysis, Reporting, Archiving
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Context:
Day 8 — Raw data ingested and profiled.
Day 9 — Data cleaned, transformed, enriched.
Today — Full analysis, final report generated,
        pipeline archived for delivery.

---

STEP 1 - Python: Full analysis script

Create: reports/analysis.py

Load enriched_data.csv from processed/.

Write the following analysis functions.
Each must return a result, not just print.

Function 1: revenue_by_specialty(df)
Returns dict: {specialty: total_fee}

Function 2: busiest_doctor(df)
Returns doctor name with most appointments.

Function 3: patient_city_summary(df)
Returns df grouped by patient city:
count of appointments and avg fee.

Function 4: fee_trend_by_day(df)
Returns df showing total fee per
day_of_week, sorted Monday to Sunday.

Function 5: top_patients(df, n=3)
Returns top n patients by total fee paid.
Use *args or **kwargs for n.

Function 6: generate_final_report(df)
Calls all above functions.
Prints a clean, formatted final report.
Saves report as reports/final_report.txt
using file write with open().

---

STEP 2 - Final report format

The final_report.txt must look like this:

HOSPITAL DATA PIPELINE — FINAL REPORT
Generated: 2024-01-20
========================================

REVENUE BY SPECIALTY
--------------------
Cardiology      : 1400
General Medicine: 550
Neurology       : 1450
Orthopedics     : 1150
Dermatology     : 300

BUSIEST DOCTOR
--------------
Dr. Carlos Torres (3 appointments)

PATIENT CITY SUMMARY
--------------------
City       Visits  Avg Fee
Delhi         2     350.0
Lagos         1     400.0
...

FEE TREND BY DAY
----------------
Wednesday : 750
Thursday  : 900
...

TOP 3 PATIENTS BY TOTAL FEE
----------------------------
1. Ana Gonzalez    — 900
2. Sneha Patel     — 1050
3. Raj Kumar       — 700

========================================
Pipeline complete. All files archived.

---

STEP 3 - SQL final verification

Run these three queries on freesql.com
and verify results match your Python report:

1. Revenue by specialty (from appointments
   joined with doctors using implicit join).

2. Patient with highest total fee paid
   (group by patient_id, sum fee).

3. Count of appointments per day of week.
   Hint: use TO_CHAR(appt_date, 'Day')
   in Oracle SQL.

All three results must match the
Python report exactly.

---

STEP 4 - Linux final archiving

1. Confirm reports/final_report.txt exists.
   cat reports/final_report.txt

2. Create the final delivery archive:
   tar -czvf hospital_pipeline_delivery.tar.gz
   hospital_pipeline/

3. Verify all expected files are inside:
   tar -tvf hospital_pipeline_delivery.tar.gz
   | grep -E "(.csv|.py|.txt|.json)"

4. Check total size:
   du -sh hospital_pipeline_delivery.tar.gz

5. Set final permissions:
   All .py files  → chmod 755
   All .csv files → chmod 644
   All .txt files → chmod 644

6. Print a final confirmation:
   echo "Pipeline delivery package ready."
   echo "Files inside:"
   tar -tvf hospital_pipeline_delivery.tar.gz
   | wc -l
   echo "files packaged."

---

End Goal:
You have built a complete three-day
data pipeline from scratch:

Day 8 → Ingested raw CSV files,
         profiled data using Pandas.

Day 9 → Cleaned all three files,
         merged them into one enriched
         dataset, verified with Linux.

Day 10 → Ran full analysis using modular
          Python functions, generated a
          formatted text report, cross-
          verified against SQL, and
          archived the entire pipeline
          for delivery.

This is a real data engineering workflow.
Same pattern used in production pipelines
at every company working with data.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DAILY TARGETS
60 minutes  -- Solve exercises
20 minutes  -- Review and discuss in group
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
'''

In [ ]:
# STEP 1 - Python: Full analysis script
import pandas as pd


def load_data(path="processed/enriched_data.csv"):
    return pd.read_csv(path)

def revenue_by_specialty(df):
    total_fee = df.groupby("specialty")["fee"].sum()
    return total_fee.to_dict()

def busiest_doctor(df):
    count = df["doctor_name"].value_counts()
    doctor_name = count.idxmax()
    appointment_count = count.max()
    return doctor_name, appointment_count

def patient_city_summary(df):
    result = df.groupby("city").agg(
        total_appointments = ("appt_id", "count"),
        avg_fee = ("fee", "mean")
    )
    return result

def fee_trend_by_day(df):
    order = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]
     
    result1 = df.groupby("day_of_week")["fee"].sum().reindex(order)
    return result1.reset_index(name="total_fee")

def top_patients(df, *args, **kwargs):
    n = kwargs.get("n", 3)

    result2 = (
        df.groupby("patient_name")["fee"].sum().sort_values(ascending=False)
        .head(n)
    )
    return result2

def generate_final_report(df):
    report_lines = []

    rev_spec = revenue_by_specialty(df)
    report_lines.append("\nRevenue by Specialty")
    report_lines.append("--------------------------------------")
    for keys, values in rev_spec.items():
        report_lines.append(f"{keys}: {values}")
    
    busiest, count = busiest_doctor(df)
    report_lines.append("\nBusiest Doctor")
    report_lines.append("--------------------------------------")
    report_lines.append(f"{busiest} ({count} appointments)")

    city_summary = patient_city_summary(df)
    report_lines.append("\nPatient City Summary")
    report_lines.append("--------------------------------------")
    report_lines.append(city_summary.to_string())

    fee_trend = fee_trend_by_day(df)
    report_lines.append("\nFee Trend by Day")
    report_lines.append("--------------------------------------")
    report_lines.append(fee_trend.to_string(index=False))

    top = top_patients(df, n=3)
    report_lines.append("\nTop Patients")
    report_lines.append("--------------------------------------")
    report_lines.append(top.to_string())

    final_report = "\n".join(report_lines)

    with open("reports/fianl_report.txt", "w") as file:
        file.write(final_report)
    
    report_lines.append("======================================")
    report_lines.append("Pipeline complete. All files archived.")

    return final_report

if __name__ == "__main__":
    df = load_data()
    generate_final_report(df)

In [ ]:
# STEP 3 - SQL final verification

# 1
SELECT 
    DOC.SPECIALIZATION,
    SUM(AP.FEE) AS TOTAL_REVENUE
FROM APPOINTMENTS AP, DOCTORS DOC
WHERE DOC.DOCTOR_ID = AP.DOCTOR_ID
GROUP BY DOC.SPECIALIZATION
ORDER BY DOC.SPECIALIZATION

# 2
SELECT 
    P.PATIENT_ID,
    SUM(AP.FEE) AS TOTAL_FEE
FROM APPOINTMENTS AP, PATIENTS P
WHERE AP.PATIENT_ID = P.PATIENT_ID
GROUP BY P.PATIENT_ID
ORDER BY TOTAL_FEE DESC
FETCH FIRST 1 ROW ONLY;

# 3
SELECT
    TO_CHAR(APPOINTMENT_DATE, 'DAY') AS DAY_OF_WEEK,
    COUNT(*) AS TOTAL_APPOINTMENTS
FROM APPOINTMENTS
    GROUP BY TO_CHAR(APPOINTMENT_DATE, 'DAY')
    ORDER BY DAY_OF_WEEK;

In [ ]:
# STEP 4 - Linux final archiving

1. cat reports/fianl_report.txt

2. tar -czvf hospital_pipeline_delivery.tar.gz hospital_pipeline/

3. tar -tvf  hospital_pipeline_delivery.tar.gz | grep -E "(.csv|.py|.txt|.json)"

4. du -sh hospital_pipeline_delivery.tar.gz

5. df -h hospital_pipeline_delivery.tar.gz

6. find . -type f -name "*.py"

7. find . -type f -name "*.py" -exec chmod 755 {} \;

8.  find . -type f -name "*.csv" -exec chmod 644 {} \;

9. find . -type f -name "*.txt" -exec chmod 644 {} \;

10. find . -type f \( -name "*.py" -o -name "*.csv" -o -name "*.txt" \) # verify permessions

11. echo "==== Pipeline Delivery Package Ready. ===="
    echo "Files Inside:"
    tar -tvf hospital_pipeline_delivery.tar.gz
    tar -tvf hospital_pipeline_delivery.tar.gz | wc -l
    echo "Files Packaged."
